In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'tatsuyananko'   # your Kaggle username
os.environ['KAGGLE_KEY'] = 'KGAT_d8dd44ff393af113eb3a2a4bb956d68c'

In [ ]:
!kaggle --version

Kaggle API 1.7.4.5


In [ ]:
!kaggle datasets download -d harishkumardatalab/food-image-classification-dataset
!unzip intel-food-image-classification-dataset.zip


Dataset URL: https://www.kaggle.com/datasets/harishkumardatalab/food-image-classification-dataset
License(s): CC0-1.0
 99% 1.66G/1.68G [00:09<00:00, 226MB/s]
100% 1.68G/1.68G [00:09<00:00, 198MB/s]
unzip:  cannot find or open intel-food-image-classification-dataset.zip, intel-food-image-classification-dataset.zip.zip or intel-food-image-classification-dataset.zip.ZIP.


In [ ]:
!ls

food-image-classification-dataset.zip  sample_data


In [ ]:
!unzip food-image-classification-dataset.zip

ストリーミング出力は最後の 5000 行に切り捨てられました。
  inflating: Food Classification dataset/idli/277.jpg  
  inflating: Food Classification dataset/idli/278.jpg  
  inflating: Food Classification dataset/idli/280.jpg  
  inflating: Food Classification dataset/idli/282.jpg  
  inflating: Food Classification dataset/idli/283.jpg  
  inflating: Food Classification dataset/idli/284.jpg  
  inflating: Food Classification dataset/idli/285.jpg  
  inflating: Food Classification dataset/idli/286.jpg  
  inflating: Food Classification dataset/idli/287.jpg  
  inflating: Food Classification dataset/idli/288.jpg  
  inflating: Food Classification dataset/idli/290.jpg  
  inflating: Food Classification dataset/idli/292.jpg  
  inflating: Food Classification dataset/idli/293.jpg  
  inflating: Food Classification dataset/idli/295.jpg  
  inflating: Food Classification dataset/idli/296.jpg  
  inflating: Food Classification dataset/idli/297.jpg  
  inflating: Food Classification dataset/idli/299.jpg  
  inflating: Foo

In [ ]:
!ls

'Food Classification dataset'		 sample_data
 food-image-classification-dataset.zip


In [ ]:
!ls "Food Classification dataset"

 apple_pie	 chicken_curry	   Fries	  kulfi         pizza
'Baked Potato'	 chole_bhature	  'Hot Dog'	  masala_dosa   samosa
 burger		'Crispy Chicken'   ice_cream	  momos         Sandwich
 butter_naan	 dal_makhani	   idli		  omelette      sushi
 chai		 dhokla		   jalebi	  paani_puri    Taco
 chapati	 Donut		   kaathi_rolls   pakode        Taquito
 cheesecake	 fried_rice	   kadai_paneer   pav_bhaji


In [ ]:
data_path = '/content/ Food Classification dataset' # '/content/  '

In [ ]:
from torchvision import datasets, transforms
#torchvision.transforms.Compose: allows you to chain multiple image transformations together to be applied sequentially as a single pipeline.
transform = transforms.Compose([
    transforms.Resize((150, 150)), # Resize the image to NxN pixels
    transforms.ToTensor() # Convert the image (PIL Image or numpy array) to a PyTorch tensor (scales pixels to [0.0, 1.0])
])

# torchvision.datasets.ImageFolder: simplifies the process of loading image data stored in a specific folder structure.simplifies the process of loading image data stored in a specific folder structure. It automatically assigns class labels based on the subfolder names
dataset = datasets.ImageFolder(
    root='/content/Food Classification dataset', # Specify the root directory and the transforms
    transform=transform # which "Pytorch" would be used
)
print("Number of classes:", len(dataset.classes))
print("Classes:", dataset.classes)
print("Total images:", len(dataset))

Number of classes: 34
Classes: ['Baked Potato', 'Crispy Chicken', 'Donut', 'Fries', 'Hot Dog', 'Sandwich', 'Taco', 'Taquito', 'apple_pie', 'burger', 'butter_naan', 'chai', 'chapati', 'cheesecake', 'chicken_curry', 'chole_bhature', 'dal_makhani', 'dhokla', 'fried_rice', 'ice_cream', 'idli', 'jalebi', 'kaathi_rolls', 'kadai_paneer', 'kulfi', 'masala_dosa', 'momos', 'omelette', 'paani_puri', 'pakode', 'pav_bhaji', 'pizza', 'samosa', 'sushi']
Total images: 23873


In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torch

root = "/content/Food Classification dataset"

train_transform = transforms.Compose([
    transforms.Resize((160, 160)),                                   # Resize the image to 160x160 pixels
    transforms.RandomResizedCrop(150),                               #torchvision.transforms.RandomResizedCrop(size, scale, ratio)
    transforms.RandomHorizontalFlip(),                             # randomly flip the image horizontally with probability p: torchvision.RandomHorizontalFlip(p)
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),                                          # Convert the image (PIL Image or numpy array) to a PyTorch tensor (scales pixels to [0.0, 1.0])
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),  # Normalize the tensor with mean and std deviation
])

val_transform = transforms.Compose([
    transforms.Resize((150,150)),
    transforms.Lambda(lambda img: img.convert("RGB")), #transforms.Lamda: a generic transform class in torchvision that allows users to apply any custom function to the input data.
    #lambda img: img.convert("RGB"): This is an anonymous Python function (a lambda function) that takes an image object (typically a PIL Image or a Tensor) as input, referred to by the variable img.
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Build TWO datasets so train/val can have different transforms
full_train = datasets.ImageFolder(
    root=root,
    transform=train_transform # which "Pytorch object" would be used
    )
full_val   = datasets.ImageFolder(
    root=root,
    transform=val_transform
    )

num_classes = len(full_train.classes)


# Same split indices for both
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_train), generator=g).tolist()
train_size = int(0.8 * len(indices))
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_ds = Subset(full_train, train_idx)
val_ds   = Subset(full_val, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # newer API
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)  # good starting LR for finetune

transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(
    root="/content/Food Classification dataset",
    transform=transform
)

num_classes = len(dataset.classes)
print("Classes:", num_classes)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 89.7MB/s]


Classes: 34


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) #MaxPool2d(2) e.g. 150 → 75

        # Input is 150x150(val_transform)
        # After 3 poolings: 150 -> 75 -> 37 -> 18 (integer floor)
        self.fc1 = nn.Linear(64 * 18 * 18, 128) #Linear(input_features, output_features). '64 * 18 * 18' comes from the shape of the feature map after the last convolution layer
        #What does 128 mean? -> That is a design choice! "compress the 20,736 features into a 128-dimensional representation"
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> [B,16,75,75]   16(output_layer/channel) × 75(size of image after pooling) × 75
        x = self.pool(F.relu(self.conv2(x)))  # -> [B,32,37,37]
        x = self.pool(F.relu(self.conv3(x)))  # -> [B,64,18,18]
        x = torch.flatten(x, 1) #Before feeding into a linear layer we flatten 64 × 18 × 18 = 20736 features int 1. So the tensor becames [BATCH_SIZE, 20736]
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN(num_classes).to(device)
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=20736, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=34, bias=True)
)


###torch.flatten
You can specify start_dim and end_dim to flatten only a specific subset of dimensions. For example, flattening a tensor of shape (10, 4, 5, 6) with start_dim=1 and end_dim=2 results in a shape of (10, 20, 6).

start_dim: The first dimension to flatten. Defaults to 0 (the beginning of the tensor).
end_dim: The last dimension to flatten. Defaults to -1 (the end of the tensor).

e.g.
mport torch

t = torch.arange(12).reshape(3, 4)
# Original tensor 't':
# tensor([[ 0,  1,  2,  3],
#         [ 4,  5,  6,  7],
#         [ 8,  9, 10, 11]])

# Flatten the entire tensor (default behavior):
t_flat = torch.flatten(t)
# Result:
# tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [ ]:
#Add Data Augmentation (Training Only)
train_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomResizedCrop(150),
    transforms.RandomHorizontalFlip(),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((150,150)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

device: cuda


In [ ]:
x, y = next(iter(train_loader))
x = x.to(device); y = y.to(device)
model.eval()
with torch.no_grad():
    logits = model(x)
print("logits mean/std:", logits.mean().item(), logits.std().item())
print(x.shape, y.shape, x.min().item(), x.max().item())

logits mean/std: 0.02828817255795002 0.7932772636413574


In [ ]:
def accuracy(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

epochs = 15

for epoch in range(1, epochs + 1):
    # ---- train ----
    model.train()
    train_loss, train_acc = 0.0, 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc += accuracy(logits, y)

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)

    # ---- val ----
    model.eval()
    val_loss, val_acc = 0.0, 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            val_loss += loss.item()
            val_acc += accuracy(logits, y)

    val_loss /= len(val_loader)
    val_acc  /= len(val_loader)

    print(f"Epoch {epoch}: "
          f"train loss {train_loss:.4f}, train acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f}, val acc {val_acc:.4f}")
    # val accuracy is greater than train accuracy: Data Augmentation Makes Training Harder / Dropout / Regularization Effects

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1: train loss 0.7077, train acc 0.7841 | val loss 0.6968, val acc 0.7995


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 2: train loss 0.6798, train acc 0.7913 | val loss 0.7604, val acc 0.7851


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

Epoch 13: train loss 0.4584, train acc 0.8584 | val loss 0.8880, val acc 0.7738


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 14: train loss 0.4496, train acc 0.8588 | val loss 0.6752, val acc 0.8214


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 15: train loss 0.4420, train acc 0.8637 | val loss 0.7530, val acc 0.8145


In [ ]:
torch.save(model.state_dict(), "resnet18_food_epoch14_val0.8214.pth")

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
import random

model.eval()

idx = random.randint(0, len(val_ds)-1)
img, label = val_ds[idx]   # use val_ds!

with torch.no_grad():
    logits = model(img.unsqueeze(0).to(device))
    pred = logits.argmax(dim=1).item()

print("True:", dataset.classes[label])
print("Pred:", dataset.classes[pred])

True: chai
Pred: chai


### How to improve accuracy more
1:Unfreeze full ResNet (if you froze it earlier)

for p in model.parameters():
    p.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

2:Increase input size slightly

IMG = 224

3: Add one augmentation

transforms.ColorJitter(brightness=0.2, contrast=0.2)